[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-biology/AdvancesMSDataProcessing/blob/master/EmergingComputationalPlatformsPipelines/googleADK.ipynb)


> **Source:** [adk.dev](https://adk.dev/)


In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00


In [ ]:
%%writefile .env
GENAI_API_KEY=''

Overwriting .env


#### **Important:** Please ensure your `GENAI_API_KEY` is set in Colab's secrets manager.

1.  Click on the "🔑" (key) icon in the left sidebar.
2.  Click "Add a new secret".
3.  Set the "Name" to `GENAI_API_KEY`.
4.  Paste your API key into the "Value" field.
5.  Toggle "Notebook access" to on.

After setting the secret, please re-run the cell above (rJjJT0Lv7fMK).

https://www.linkedin.com/pulse/how-build-simple-ai-agent-46-lines-python-code-morgan-linton-ogqec

Here's what the loop does 🔽

    Model gets your question
    It decides: "I need a tool"
    Tool runs, result goes back in
    Model sees the result, decides next step
    Stops when it has the answer

The key thing to understand here is that the decision loop is the agent.


Python Quickstart for ADK

https://adk.dev/get-started/python/

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key="")

# 1. Clear function definition (Gemini extracts schema from this)
def calculator(expression: str) -> str:
    """Evaluates a math expression.

    Args:
        expression: The mathematical expression to compute.
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

# Route execution mapping based on the function name Gemini chooses
def run_tool(name, inputs):
    if name == "calculator":
        return calculator(**inputs)

config = types.GenerateContentConfig(
    max_output_tokens=1024,
    tools=[calculator],
)

def run_agent(user_message):
    # Gemini conversation maps as a list of types.Content objects
    messages = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_message)]
        )
    ]

    while True:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=messages,
            config=config
        )

        # 3. If there are no tool execution requests, we are done
        if not response.function_calls:
            return response.text

        # Keep track of tool responses for Gemini's structural turn requirement
        tool_response_parts = []

        # Loop directly over Gemini's native function call structures
        for function_call in response.function_calls:
            # Execute the tool safely passing args as a dict
            output = run_tool(function_call.name, function_call.args)

            # Formulate the explicit confirmation match Gemini requires
            tool_response_parts.append(
                types.Part.from_function_response(
                    name=function_call.name,
                    # Output must always be passed nested inside a response map dictionary
                    response={"result": output}
                )
            )

        # Chronological turn synchronization:
        # A Turn consists of appending what the model decided (the calls)...
        messages.append(response.candidates[0].content)

        # ...followed immediately by your system responses to those exact calls
        messages.append(
            types.Content(
                role="user",
                parts=tool_response_parts
            )
        )


In [ ]:
print(run_agent("What is (137 * 48) + (256 / 16)?"))

The result of (137 * 48) + (256 / 16) is 6592.0.
